# A2g Vehicle 3D geometry fine-tune gate

This notebook starts from the verified A2 MobileNetV4 Conv Medium epoch-130 checkpoint. It runs three controlled 20-epoch branches:

- `control_w1_0`: unchanged A2 loss coefficients
- `geometry_w1_5`: 1.5x 3D-center, dimension, yaw, and depth losses
- `geometry_w2_0`: 2.0x the same geometry losses

The notebook does not use distillation, change the architecture, change matching, or change sampling. Run the training cell once per branch by changing `RUN_VARIANT`; it resumes from the latest complete checkpoint if the Colab session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from collections import deque
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, re, shlex, shutil, subprocess, sys

MOBILE_REPO = Path('/content/mobile_adas3d')
MONODETR_REPO = Path('/content/MonoDETR')
MONODETR_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
LOCAL_DATASET_ROOT = Path('/content/kitti')
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
MONODETR_KITTI = Path('/content/monodetr_kitti_a2g')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
R0_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
A2_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_gt/product_checkpoint_sweep/a2_product_selection.json')
BASE_CONFIG = MONODETR_REPO / 'configs/monodetr_a2_mnv4_vehicle_pedestrian_gt.yaml'
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2g_vehicle_geometry_gate')
GATE_EPOCHS = 20
LEARNING_RATE = 1e-5
RUN_VARIANT = 'control_w1_0'  # change to geometry_w1_5 or geometry_w2_0 for the next branch
A2_EPOCH = 130
A2_SHA256 = 'ed2134a98acbf1ab2fc61f7c8749b38fdfd2418e7f7932593e5e37a8d9ef33f4'
VARIANTS = {'control_w1_0': 1.0, 'geometry_w1_5': 1.5, 'geometry_w2_0': 2.0}
GEOMETRY_KEYS = ('3dcenter_loss_coef', 'dim_loss_coef', 'angle_loss_coef', 'depth_loss_coef')
assert RUN_VARIANT in VARIANTS

def run(command, cwd=None, env=None):
    command = [str(x) for x in command]
    print('+', shlex.join(command), flush=True)
    merged = os.environ.copy(); merged.update(env or {})
    result = subprocess.run(command, cwd=cwd, env=merged)
    if result.returncode:
        raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

A2_SELECTION.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Selected branch:', RUN_VARIANT)

In [ ]:
# Prepare the pinned MonoDETR environment and the symlinked Chen KITTI view.
if not MOBILE_REPO.exists():
    run(['git', 'clone', 'https://github.com/Ali-RT/mobile_adas3d.git', MOBILE_REPO])
if not MONODETR_REPO.exists():
    run(['git', 'clone', 'https://github.com/ZrrSkywalker/MonoDETR.git', MONODETR_REPO])
run(['git', 'fetch', '--all'], cwd=MONODETR_REPO)
run(['git', 'checkout', MONODETR_COMMIT], cwd=MONODETR_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'pyyaml', 'scipy', 'opencv-python-headless', 'numba', 'scikit-image', 'tqdm', 'ninja', 'timm==1.0.20', 'pandas'])
for patch in ('patch_monodetr_colab_compat.py', 'patch_monodetr_product_taxonomy.py', 'patch_monodetr_mobilenetv4.py', 'patch_monodetr_verbose_resume.py', 'patch_monodetr_checkpoint_metadata.py'):
    run([sys.executable, f'scripts/{patch}', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
ops = MONODETR_REPO / 'lib/models/monodetr/ops'
shutil.rmtree(ops / 'build', ignore_errors=True)
run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops, env={'MAX_JOBS': '2'})
run([sys.executable, '-c', "import torch,timm,MultiScaleDeformableAttention; assert timm.__version__ == '1.0.20'; print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))"], cwd=MONODETR_REPO)

def resolve(root, names):
    for name in names:
        candidate = root / name
        if candidate.is_dir(): return candidate
    return None

sources = {}
for key, names in {
    'image_2': ['training/image_2', 'training/image_02'],
    'label_2': ['training/label_2', 'training/label_02'],
    'calib': ['training/calib'],
}.items():
    sources[key] = resolve(LOCAL_DATASET_ROOT, names) or resolve(DRIVE_DATASET_ROOT, names)
if any(value is None for value in sources.values()):
    raise FileNotFoundError(f'Missing KITTI sources: {sources}')
(MONODETR_KITTI / 'training').mkdir(parents=True, exist_ok=True)
(MONODETR_KITTI / 'ImageSets').mkdir(parents=True, exist_ok=True)
for name, target in sources.items():
    link = MONODETR_KITTI / 'training' / name
    if link.is_symlink() and link.resolve() == target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target, target_is_directory=True)
for split in ('train', 'val'):
    shutil.copy2(SPLIT_DIR / f'{split}.txt', MONODETR_KITTI / 'ImageSets' / f'{split}.txt')
assert len((MONODETR_KITTI / 'ImageSets/val.txt').read_text().splitlines()) == 3769

# The A2 YAML is generated by our preparation step, not tracked by upstream MonoDETR.
if not A2_SELECTION.is_file(): raise FileNotFoundError(f'Missing A2 selection; run the A2 baseline first: {A2_SELECTION}')
if not BASE_CONFIG.is_file():
    print('A2 config is not tracked upstream; recreating the deterministic A2 config...')
    run([
        sys.executable, 'scripts/prepare_monodetr_a2_student.py',
        '--monodetr-repo', MONODETR_REPO,
        '--dataset-root', MONODETR_KITTI,
        '--r0-selection', R0_SELECTION,
        '--output-root', Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_gt'),
        '--run-name', 'monodetr_a2_mnv4_vehicle_pedestrian_gt',
        '--max-epochs', '195', '--save-frequency', '5',
        '--batch-size', '16', '--learning-rate', '1e-4',
    ], cwd=MOBILE_REPO)
if not BASE_CONFIG.is_file(): raise FileNotFoundError(BASE_CONFIG)
selection = json.loads(A2_SELECTION.read_text())
assert selection.get('complete') and int(selection['selected_epoch']) == A2_EPOCH
A2_CHECKPOINT = Path(selection['selected_checkpoint'])
assert A2_CHECKPOINT.is_file(), A2_CHECKPOINT
assert sha256(A2_CHECKPOINT) == A2_SHA256, 'A2 epoch-130 checkpoint hash mismatch'
print('Verified A2 checkpoint:', A2_CHECKPOINT)

In [ ]:
# Generate the three paired configs and a provenance manifest.
import yaml
base = yaml.safe_load(BASE_CONFIG.read_text())
base['dataset'].update({'root_dir': str(MONODETR_KITTI), 'train_split': 'train', 'test_split': 'val'})
configs = {}
for variant, multiplier in VARIANTS.items():
    cfg = yaml.safe_load(yaml.safe_dump(base))
    run_name = f'monodetr_a2g_{variant}_gate{GATE_EPOCHS}'
    cfg['model_name'] = run_name
    cfg['random_seed'] = 20268
    originals = {}
    for key in GEOMETRY_KEYS:
        if key not in cfg['model']: raise KeyError(f'Missing model.{key}')
        originals[key] = float(cfg['model'][key])
        cfg['model'][key] = originals[key] * multiplier
    cfg['optimizer']['lr'] = LEARNING_RATE
    run_dir = OUTPUT_ROOT / run_name
    cfg['trainer'].update({
        'max_epoch': GATE_EPOCHS, 'save_frequency': 5, 'save_all': True,
        'save_path': os.path.relpath(OUTPUT_ROOT, MONODETR_REPO),
        'pretrain_model': str(A2_CHECKPOINT), 'resume_model': False, 'log_frequency': 20,
    })
    cfg['tester'].update({'mode': 'single', 'checkpoint': GATE_EPOCHS, 'threshold': 0.001, 'topk': 50})
    config_path = MONODETR_REPO / f'configs/{run_name}.yaml'
    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    configs[variant] = {'run_name': run_name, 'config': str(config_path), 'run_dir': str(run_dir), 'multiplier': multiplier, 'originals': originals}

manifest = {
    'schema_version': 1, 'complete': True,
    'experiment': 'A2g paired Vehicle 3D geometry-loss gate',
    'a2_epoch': A2_EPOCH, 'a2_checkpoint': str(A2_CHECKPOINT), 'a2_checkpoint_sha256': A2_SHA256,
    'gate_epochs': GATE_EPOCHS, 'learning_rate': LEARNING_RATE,
    'geometry_keys': list(GEOMETRY_KEYS), 'distillation_enabled': False,
    'architecture_changed': False, 'backbone_changed': False, 'matcher_changed': False,
    'sampling_changed': False, 'variants': configs,
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'a2g_gate_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(json.dumps(manifest, indent=2))

In [ ]:
# Train one branch. Change RUN_VARIANT in the first cell and rerun this cell for the other branches.
variant = RUN_VARIANT
run_name = configs[variant]['run_name']
run_dir = Path(configs[variant]['run_dir'])
config_path = Path(configs[variant]['config'])
checkpoint_pattern = re.compile(r'^checkpoint_epoch_(\d+)\.pth$')
existing = []
for path in run_dir.glob('checkpoint_epoch_*.pth'):
    match = checkpoint_pattern.match(path.name)
    if not match: continue
    try:
        payload = __import__('torch').load(path, map_location='cpu', weights_only=False)
        if int(payload.get('epoch', -1)) == int(match.group(1)) and payload.get('optimizer_state') is not None:
            existing.append((int(match.group(1)), path))
    except Exception as error:
        print('Skipping invalid checkpoint', path, type(error).__name__, error)
latest = max(existing, default=None, key=lambda item: item[0])
cfg = yaml.safe_load(config_path.read_text())
if latest is None:
    cfg['trainer'].pop('resume_model', None)
    cfg['trainer']['pretrain_model'] = str(A2_CHECKPOINT)
    print('Starting', variant, 'from A2 epoch 130')
elif latest[0] >= GATE_EPOCHS:
    print(variant, 'already complete at epoch', latest[0])
else:
    cfg['trainer'].pop('pretrain_model', None)
    cfg['trainer']['resume_model'] = str(latest[1])
    print('Resuming', variant, 'from epoch', latest[0], latest[1])
runtime_config = MONODETR_REPO / f'configs/{run_name}_runtime.yaml'
runtime_config.write_text(yaml.safe_dump(cfg, sort_keys=False))

if latest is None or latest[0] < GATE_EPOCHS:
    log_dir = OUTPUT_ROOT / 'colab_logs'; log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f'train_{run_name}.log'
    env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
    command = [sys.executable, '-u', 'tools/train_val.py', '--config', runtime_config]
    print('+', shlex.join(map(str, command)), '\nDurable log:', log_path, flush=True)
    with log_path.open('a', buffering=1) as log:
        process = subprocess.Popen([str(x) for x in command], cwd=MONODETR_REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end=''); log.write(line)
        code = process.wait()
    if code: raise RuntimeError(f'{variant} training failed; durable log: {log_path}')
else:
    print('No training required.')

In [ ]:
# Sweep the selected branch and apply the frozen five product gates.
variant = RUN_VARIANT
run_name = configs[variant]['run_name']
run_dir = Path(configs[variant]['run_dir'])
sweep_dir = run_dir / 'product_checkpoint_sweep'
training_config = Path(configs[variant]['config'])
run([sys.executable, '-u', 'scripts/sweep_monodetr_r0_product_checkpoints.py',
     '--monodetr-repo', MONODETR_REPO, '--mobile-repo', MOBILE_REPO,
     '--training-config', training_config, '--run-dir', run_dir,
     '--dataset-root', MONODETR_KITTI, '--split-dir', SPLIT_DIR,
     '--output-dir', sweep_dir, '--product-config', 'configs/kitti_mobileadas3d_s1.yaml',
     '--profile', 'colab_drive', '--score-threshold', '0.001', '--topk', '50',
     '--source-name-prefix', 'MobileMonoDETR_A2g'], cwd=MOBILE_REPO)
raw = json.loads((sweep_dir / 'r0_product_selection.json').read_text())
selected = raw['selected']
gates = {'vehicle_3d_moderate': 15.8713, 'pedestrian_3d_moderate': 5.1493, 'mean_3d_moderate': 10.5103, 'vehicle_bev_moderate': 21.3134, 'pedestrian_bev_moderate': 5.9365}
gate_results = {key: float(selected[key]) >= value for key, value in gates.items()}
report = {'schema_version': 1, 'complete': bool(raw.get('complete')), 'variant': variant, 'selected_epoch': selected['epoch'], 'selected_checkpoint': selected['checkpoint'], 'metrics': {key: float(selected[key]) for key in gates}, 'gates': gates, 'gate_results': gate_results, 'all_accuracy_gates_passed': all(gate_results.values()), 'nearby_recall_review_required': True}
(sweep_dir / 'a2g_product_selection.json').write_text(json.dumps(report, indent=2) + '\n')
print(json.dumps(report, indent=2))

In [ ]:
# Compare completed branches. Missing branches are shown as pending.
import pandas as pd
rows = []
for variant, info in configs.items():
    report_path = Path(info['run_dir']) / 'product_checkpoint_sweep' / 'a2g_product_selection.json'
    if report_path.is_file():
        report = json.loads(report_path.read_text())
        row = {'variant': variant, 'selected_epoch': report['selected_epoch'], **report['metrics'], 'all_gates_passed': report['all_accuracy_gates_passed']}
    else:
        row = {'variant': variant, 'status': 'pending'}
    rows.append(row)
display(pd.DataFrame(rows))